# A04 - 共轭传热 CHT：流体-固体界面条件

区域被分为上下两层：

- 流体区：`0 <= y <= 0.5`，给定速度 `u(y)`，求 `T_f`
- 固体区：`0.5 <= y <= 1`，纯导热，求 `T_s`

界面 `y=0.5` 满足：

- 温度连续：`T_f = T_s`
- 热流连续：`k_f*dT_f/dn = k_s*dT_s/dn`

这就是共轭传热 PINN 的核心。后续可以把流体速度换成 N-S 网络输出。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(104)
device = "cuda" if torch.cuda.is_available() else "cpu"
Pe_f = 15.0
k_f, k_s = 1.0, 5.0
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=64, depth=5):
        super().__init__()
        layers = [nn.Linear(2, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        return self.net(xy)

def grad(y, x):
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=True)[0]

Tf_net = MLP().to(device)
Ts_net = MLP().to(device)

def u_fluid(y):
    eta = y / 0.5
    return 4 * eta * (1 - eta)

In [ ]:
def fluid_residual(x, y):
    x.requires_grad_(True); y.requires_grad_(True)
    T = Tf_net(torch.cat([x, y], dim=1))
    T_x, T_y = grad(T, x), grad(T, y)
    T_xx, T_yy = grad(T_x, x), grad(T_y, y)
    return u_fluid(y) * T_x - (T_xx + T_yy) / Pe_f

def solid_residual(x, y):
    x.requires_grad_(True); y.requires_grad_(True)
    T = Ts_net(torch.cat([x, y], dim=1))
    T_x, T_y = grad(T, x), grad(T, y)
    T_xx, T_yy = grad(T_x, x), grad(T_y, y)
    return T_xx + T_yy

def interface_loss(n=200):
    x = torch.rand(n, 1, device=device)
    y = 0.5 * torch.ones_like(x)
    y.requires_grad_(True)
    Tf = Tf_net(torch.cat([x, y], dim=1))
    Ts = Ts_net(torch.cat([x, y], dim=1))
    Tf_y = grad(Tf, y)
    Ts_y = grad(Ts, y)
    loss_T = torch.mean((Tf - Ts)**2)
    loss_q = torch.mean((k_f * Tf_y - k_s * Ts_y)**2)
    return loss_T + loss_q

def external_bc_loss(n=160):
    s = torch.rand(n, 1, device=device)
    Tf_in = Tf_net(torch.cat([torch.zeros_like(s), 0.5 * s], dim=1))
    loss_in = torch.mean((Tf_in - 1.0)**2)
    Ts_top = Ts_net(torch.cat([s, torch.ones_like(s)], dim=1))
    loss_top = torch.mean(Ts_top**2)
    Tf_bottom = Tf_net(torch.cat([s, torch.zeros_like(s)], dim=1))
    loss_bottom = torch.mean(Tf_bottom**2)
    return loss_in + loss_top + loss_bottom

In [ ]:
optimizer = torch.optim.Adam(list(Tf_net.parameters()) + list(Ts_net.parameters()), lr=1e-3)
history = []
for epoch in range(10000):
    xf = torch.rand(768, 1, device=device)
    yf = 0.5 * torch.rand(768, 1, device=device)
    xs = torch.rand(768, 1, device=device)
    ys = 0.5 + 0.5 * torch.rand(768, 1, device=device)
    loss_pde = torch.mean(fluid_residual(xf, yf)**2) + torch.mean(solid_residual(xs, ys)**2)
    loss_if = interface_loss()
    loss_bc = external_bc_loss()
    loss = loss_pde + 20 * loss_if + 10 * loss_bc
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    history.append([loss.item(), loss_pde.item(), loss_if.item(), loss_bc.item()])
    if epoch % 2000 == 0:
        print(f"epoch={epoch:5d}, total={loss.item():.3e}, pde={loss_pde.item():.3e}, interface={loss_if.item():.3e}")

In [ ]:
nx, ny = 100, 100
xs = torch.linspace(0, 1, nx, device=device)
ys = torch.linspace(0, 1, ny, device=device)
X, Y = torch.meshgrid(xs, ys, indexing="ij")
xy = torch.cat([X.reshape(-1,1), Y.reshape(-1,1)], dim=1)
with torch.no_grad():
    T = torch.empty(nx * ny, 1, device=device)
    mask = xy[:, 1:2] <= 0.5
    T[mask[:,0]] = Tf_net(xy[mask[:,0]])
    T[~mask[:,0]] = Ts_net(xy[~mask[:,0]])
    T = T.reshape(nx, ny)

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.imshow(T.cpu().T, origin="lower", extent=[0,1,0,1], aspect="auto")
plt.axhline(0.5, color="white", linestyle="--")
plt.colorbar(); plt.title("CHT temperature")
plt.subplot(1, 2, 2)
plt.semilogy(torch.tensor(history)[:,0], label="total")
plt.semilogy(torch.tensor(history)[:,2], label="interface")
plt.legend(); plt.title("loss")
plt.tight_layout(); plt.show()